# BN calibration and power
Uses your original BN function unchanged. Every dataset has **5 hosts**, **50 or 100 variants**, and **200 inner bootstrap replicates**. Read README.md for model assumptions and limitations. No study results are precomputed here.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
import run_validation as study
assert study.N_BOOT == 200 and study.N_HOSTS == 5
pd.DataFrame(study.SCENARIOS)

## Choose outer replication
Start with a pilot to measure runtime. Set PILOT=False for 200 independent datasets per scenario. These are outer datasets, distinct from the 200 inner bootstrap replicates. Use the same settings and folder to resume. A full run may be lengthy.

In [ ]:
PILOT = True
N_DATASETS = 1 if PILOT else 200
SCENARIOS = ["null_small"] if PILOT else None
OUTPUT = "pilot_results" if PILOT else "validation_results"
N_JOBS = 2
SEED = 12345

In [ ]:
calibration, bh_metrics = study.run(
    n_datasets=N_DATASETS, output=OUTPUT, seed=SEED,
    n_jobs=N_JOBS, scenario_names=SCENARIOS)

## Inspect completion and failures first
Incomplete/failed runs must not be silently excluded from manuscript conclusions.

In [ ]:
display(pd.read_csv(Path(OUTPUT)/"completion.csv"))
display(calibration)
display(bh_metrics)

In [ ]:
display(Image(filename=str(Path(OUTPUT)/"null_calibration.png")))
feature_calibration = pd.read_csv(Path(OUTPUT)/"calibration_by_feature.csv")
display(feature_calibration.head(20))

## Interpretation
Compare unadjusted null rejection rates with nominal levels, including Monte Carlo uncertainty. Featurewise exact intervals use independent outer datasets. Dataset-average SEs account for within-dataset dependence by aggregating first. Under the complete null, BH FDR is the probability of any rejection. Power/FDR in mixed scenarios use the formal truth q_j=D_j. With B=200, p>=1/201, which limits BH resolution. Zero rejections alone do not demonstrate usefulness or exact validity. These scenarios assess calibration under the assumed generating model, not robustness to biological misspecification. Report all scenarios and do not infer general validity from successful simulations.